# Week 2, Day 1: RAG architecture, a document assistant

RAG means: **fetch your own text first, then answer from it**. Three stages, kept separate: **ingest** (cut documents into chunks), **retrieve** (find the best chunks), **generate** (write a cited answer, or say you do not know). The project did this in C#. Here it is in plain Python with no downloads: chunks, a simple word-count similarity, and an extractive answer.

Run the cells from the top, one at a time (Shift+Enter).

## Step 1: ingest
Four small documents. Each is cut into paragraph chunks that remember their document and part number (the *source reference*).

In [ ]:
DOCS = {
 "Leave policy":      "Every full-time employee gets 24 days of paid annual leave per calendar year.\n\nUnused leave carries over up to 10 days and expires on 31 March.\n\nSick leave is separate from annual leave.",
 "Remote work and security": "Employees may work remotely up to 3 days a week after probation.\n\nCompany laptops must use full disk encryption and lock automatically after 5 minutes.\n\nUse the VPN on any public Wi-Fi.",
 "Expenses":          "Business expenses are reimbursed within 14 days of a complete claim.\n\nAny single purchase above 500 euros needs written approval from a director before you buy.",
 "Onboarding and training": "Each employee has a learning budget of 1,500 euros per year for courses, books and conferences.\n\nNew hires are paired with a mentor for their first 3 months.",
}

chunks = []
for title, body in DOCS.items():
    for part, text in enumerate(body.split("\n\n"), start=1):
        chunks.append({"title": title, "part": part, "text": text})
print(len(chunks), "chunks")
print(chunks[3])

## Step 2: retrieve
Simple and readable: score each chunk by how many question words it shares (ignoring common words), then take the best K. Real systems use vectors, but the stage is the same: *question in, best chunks out*.

In [ ]:
import re
STOP = {"the","a","an","of","to","is","are","do","does","how","what","when","who","i","my","can","or","and","in","for","after","many","much","it","does","get"}
def words(t): return {w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOP}

def retrieve(question, k=3):
    q = words(question)
    scored = sorted(((len(q & words(c["text"])), i) for i, c in enumerate(chunks)), reverse=True)
    return [chunks[i] | {"score": s} for s, i in scored[:k] if s > 0]

for c in retrieve("After how many minutes does a company laptop lock itself?"):
    print(c["score"], c["title"], "part", c["part"], "|", c["text"][:60])

## Step 3: generate, with a citation and a fallback
Pick the best sentence from the retrieved chunks and cite its number [n]. If nothing covers the question, say so instead of guessing. (The project asks a model to write the answer from the numbered chunks and only accepts it if every [n] is real; the extractive version here is the safe fallback.)

In [ ]:
FALLBACK = "I could not find this in the approved documents."
def answer(question, k=3):
    hits = retrieve(question, k)
    q = words(question)
    best, best_n, best_score = None, None, 0
    for n, c in enumerate(hits, start=1):
        for sentence in re.split(r"(?<=[.!?])\s+", c["text"]):
            s = len(q & words(sentence))
            if s > best_score: best, best_n, best_score = sentence, n, s
    if best is None or best_score < 2:          # not enough overlap: refuse
        return {"answer": FALLBACK, "sources": []}
    return {"answer": f"{best} [{best_n}]", "sources": [f"[{n}] {c['title']}, part {c['part']}" for n, c in enumerate(hits, start=1)]}

for q in ["After how many minutes does a company laptop lock itself?", "How soon are expenses reimbursed?", "How much is the annual bonus?"]:
    print(q, "\n  ->", answer(q), "\n")

## Step 4: practise
1. The bonus question is refused. Change the `best_score < 2` rule to `< 1`. What happens to the bonus question? What does that teach about the fallback?
2. Add a fifth document and ask about it.
3. Write down which stage you would look at first if an answer is wrong.

**Where this is in the project:** `RagEngine.cs` (chunker, retriever, answerer), `docs/evidence/week2-day1.md`.